# OCR de rótulos y texto en pantalla

Versión mínima: leemos el vídeo, pasamos [EasyOCR](https://github.com/JaidedAI/EasyOCR) a un fotograma de cada `step` y guardamos qué texto aparece, cuándo y dónde. Al final dibujamos el resultado sobre el vídeo.

In [1]:
import cv2, subprocess, unicodedata
import numpy as np
import pandas as pd
import easyocr
from IPython.display import Video, display

VIDEO = "videos/robot_social_CSIC.mp4"     # <- cambiad aquí para usar otro vídeo
cap = cv2.VideoCapture(VIDEO)
FPS, W, H = cap.get(cv2.CAP_PROP_FPS), int(cap.get(3)), int(cap.get(4))
print(f"{FPS:.0f} fps, {W}x{H}, {cap.get(cv2.CAP_PROP_FRAME_COUNT) / FPS:.0f} s")

25 fps, 640x360, 149 s


## 1. OCR sobre los fotogramas muestreados

Un rótulo dura varios segundos, así que no hace falta leer todos los fotogramas: con `SAMPLE_FPS = 1` leemos uno por segundo. Para leerlos todos, poned `SAMPLE_FPS = FPS` (es mucho más lento).

`reader.readtext(frame)` devuelve, para cada texto encontrado, `(caja, texto, confianza)`. La caja son las 4 esquinas del texto.

In [2]:
SAMPLE_FPS = 1                            # fotogramas por segundo que pasamos por el OCR
MIN_CONF = 0.4                            # descartamos lecturas con menos confianza
step = max(1, round(FPS / SAMPLE_FPS))    # cada cuántos fotogramas pasamos el OCR

reader = easyocr.Reader(["es", "en"])     # la primera vez descarga los modelos; usa la GPU si hay

rows = []
cap = cv2.VideoCapture(VIDEO)
i = 0                                     # número de fotograma
while True:
    ok, frame = cap.read()
    if not ok: break
    if i % step == 0:
        for box, text, conf in reader.readtext(frame):     # frame en BGR, como lo lee OpenCV
            if conf >= MIN_CONF:
                rows.append(dict(frame=i, t=i / FPS, text=text, conf=conf, box=box))
    i += 1

ocr = pd.DataFrame(rows, columns=["frame", "t", "text", "conf", "box"])
print(len(ocr), "textos leídos en", (i - 1) // step + 1, "fotogramas analizados")
ocr[["t", "text", "conf"]].round(2)

137 textos leídos en 150 fotogramas analizados


,t,text,conf
0,0.0,CSIC,0.45
1,1.0,CSIC,0.89
2,2.0,CSIC,0.46
3,3.0,CSIC,0.46
4,4.0,CSIC,0.46
...,...,...,...
132,145.0,CSIC,1.00
133,146.0,CSIC,1.00
134,147.0,CSIC,1.00
135,148.0,CSIC,1.00


## 2. Ver el resultado en el vídeo

Sobre cada fotograma dibujamos las cajas y el texto leídos en el último fotograma analizado (amarillo, con la confianza entre paréntesis).

In [4]:
by_frame = ocr.groupby("frame")
cap = cv2.VideoCapture(VIDEO)
vw = cv2.VideoWriter("ocr_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
current = []                              # lecturas del último fotograma analizado
i = 0
while True:
    ok, frame = cap.read()
    if not ok: break
    if i % step == 0:
        current = by_frame.get_group(i).to_dict("records") if i in by_frame.groups else []
    for r in current:
        pts = np.array(r["box"], dtype=np.int32)          # 4 esquinas del texto
        cv2.polylines(frame, [pts], True, (0, 255, 255), 2)
        # OpenCV no sabe dibujar tildes ni ñ: las quitamos solo para el dibujo
        label = unicodedata.normalize("NFKD", r["text"]).encode("ascii", "ignore").decode()
        cv2.putText(frame, f"{label} ({r['conf']:.2f})", (int(pts[0][0]), max(int(pts[0][1]) - 5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    vw.write(frame)
    i += 1
vw.release()

# convert to H.264 so that the browser can play it
subprocess.run("ffmpeg -y -loglevel error -i ocr_raw.mp4 -vcodec libx264 -pix_fmt yuv420p ocr_annotated.mp4", shell=True)
display(Video("ocr_annotated.mp4", embed=True, width=640))

In [13]:
ocr.to_csv("dataframes/ocr_csic.tsv", index=False, sep="\t", encoding="utf-8") 